In [27]:
import pandas as pd
import pyomo.environ as pyo
from itertools import product
from pyomo.environ import value
import numpy as np
import random
import math

model = pyo.ConcreteModel()


# 1. SETS

T=[0, 1, 2, 3, 4, 5]  # horizon temporel

#définition des zones
dA = pd.read_csv("AREAS.csv", sep=";")  
random.seed(4)
na  = 30              #nombre de zones de la petite instance
n = len(dA["AREAS"])                            #nombre de zones au total
A = [dA["AREAS"][i] for i in random.sample(range(0, n), na) ]



I=['ORANGE','FREE MOBILE', 'BOUYGUES TELECOM','SFR']  # opérateurs
τ='ORANGE'  # notre opérateur
O=['o3G-FREE MOBILE', 'o3G-BOUYGUES TELECOM', 'o3G-SFR','o3G-ORANGE','o4G-FREE MOBILE','o4G-BOUYGUES TELECOM','o4G-SFR','o4G-ORANGE','o5G-FREE MOBILE','o5G-BOUYGUES TELECOM','o5G-SFR','o5G-ORANGE']  
O_i = {"ORANGE" : ['o3G-ORANGE' , 'o4G-ORANGE', 'o5G-ORANGE'] , 'FREE MOBILE' : ['o3G-FREE MOBILE', 'o4G-FREE MOBILE', 'o5G-FREE MOBILE'] ,
      'BOUYGUES TELECOM' : ['o3G-BOUYGUES TELECOM', 'o4G-BOUYGUES TELECOM', 'o5G-BOUYGUES TELECOM'] , 'SFR' : ['o3G-SFR', 'o4G-SFR', 'o5G-SFR']  }# offres

NG = 'o5G-ORANGE'

model.T = pyo.Set(initialize=T)  # horizon temporel
model.A = pyo.Set(initialize=A)  # zones
model.I = pyo.Set(initialize=I)  # opérateurs
model.O = pyo.Set(initialize=O)  # offres !!! il faut mettre l'offre de chaque opérateur
model.O_i = O_i




In [28]:
### Couples utiles
C_space = list(product([0,1], repeat=len(I)))  # Toutes les combinaisons de couverture
df = pd.read_csv("AREAS_SITES_LINK.csv", sep=";")   # contient colonnes "a" et "s"

Sa_dict = {}
S = [] 

i = 0
for _, row in df.iterrows():
    zone = row["AREAS"]
    site = row["SITES"]
    if zone not in A:             #on ne considère que les zones sélectionnées dans A
        continue

    if zone not in Sa_dict :
        Sa_dict[zone] = []

    if site not in Sa_dict[zone]:     #éviter d'ajouter plusieurs fois la même zone
        Sa_dict[zone].append(site)
        
    if site not in S:
        S.append(site)

model.S = pyo.Set(initialize=S) # sites de l’opérateur i

As_dict = {}

for _, row in df.iterrows():
    zone = row["AREAS"]
    site = row["SITES"]
    if site not in S:
        continue

    if site not in As_dict:
        As_dict[site] = []
    if zone in A and zone not in As_dict[site]:
        As_dict[site].append(zone)



def _C_tuple(m, t, a, r_tau_val):
    C_list = [r_tau_val]
    autres_operateurs = list(m.I)[1:] # On respecte ton indexation (tau est à l'index 0)
    for i in autres_operateurs:
        C_list.append(pyo.value(m.Rcomp[t, a, i]))
    return tuple(C_list)

    


model.Cvec = pyo.Set(initialize=C_space)   # Toutes les combinaisons de couverture
model.Sa = Sa_dict                         # mapping a → {sites}
model.As = As_dict                         # mapping s → {zones}

# 2. PARAMÈTRES
df_zmax = pd.read_csv("OPERATIONAL_LIMITS.csv", sep=";")
Zmax_data = [0, 5, 5, 5, 5, 5]                            #modifiable
model.Zmax = pyo.Param(model.T, initialize = Zmax_data)  # nombre max de sites déployables par période

df_qa = pd.read_csv("STRATEGIC_GUIDELINES.csv", sep=";")
QA_data = {row.TIME_SLOTS: row["QA"] for _,row in df_qa.iterrows()}
QA_data[0] = 0
model.QA = pyo.Param(model.T, initialize = QA_data) # couverture minimale de la population par période

df = pd.read_csv("AREAS.csv", sep=";")

ua0_data = {}
for _, row in df.iterrows():
    for col in df.columns [1:]:
        if row["AREAS"] in A :                      #on ne prend que les aires de la petite instance
            ua0_data[row["AREAS"],col.split("-")[1], col] = row[col]    #partie entière retirée
for a in A:
    for i in I:
        for o in O:
            if (a,i,o) not in ua0_data:
                ua0_data[a,i,o] = 0

    
        
model.ua0 = pyo.Param(model.A, model.I, model.O, initialize=ua0_data)  # utilisateurs initiaux

df_dng = pd.read_csv("DEMAND.csv", sep=";")

DNG_data = {row.TIME_SLOTS: row["5G"] for _,row in df_dng.iterrows()}
DNG_data[0] = 0
model.DNG = pyo.Param(model.T, initialize = DNG_data)  # DNG dépend du temps

df_capang = pd.read_csv("CAPACITY.csv", sep=";")
CAPANG_data = {row.TIME_SLOTS: row["5G"] for _,row in df_capang.iterrows()}
CAPANG_data[0] = 0
model.CAPANG = pyo.Param(model.T, initialize = CAPANG_data) # DNG et CAPANG dépendent du temps

df_ua = pd.read_csv("AREAS.csv", sep=";")
u_a_data = {}
for _, row in df.iterrows():
    if row["AREAS"] in A :
        somme =0
        for col in df.columns [1:]:
            somme += row[col]
        u_a_data [row["AREAS"]]= somme

model.u_a = pyo.Param(model.A, initialize = u_a_data)  # utilisateurs totaux dans la zone a

df_Rcomp = pd.read_csv("COMPETITORS_STRATEGY.csv", sep=";")
Rcomp_data = {}
for _, row in df_Rcomp.iterrows():
    if row["AREAS"] in A :
        for col in df_Rcomp.columns[2:5]:
            Rcomp_data[row["TIME_SLOTS"],row["AREAS"], col] = 1 * row[col]
model.Rcomp = pyo.Param(model.T, model.A, model.I, initialize=Rcomp_data)  # couverture des autres opérateurs

#f

df_fdata = pd.read_csv("UPGRADE_FUNCTION.csv", sep=";")

f_data = {}
for a in A:
    for _, row in df_fdata.iterrows():
        C  = (row["ORANGE"], row["FREE MOBILE"], row["BOUYGUES TELECOM"], row["SFR"])
        O1 = row["OFFERS"] + "-" + row["FROM_OPERATOR"]
        O2 = "o5G-" + row["TO_OPERATOR"]
        f_data[(a, C, O1, O2)] = row["PERCENTAGES"]

O_full= ['o3G-FREE MOBILE', 'o3G-BOUYGUES TELECOM', 'o3G-SFR', 'o3G-ORANGE',
       'o4G-FREE MOBILE', 'o4G-BOUYGUES TELECOM', 'o4G-SFR', 'o4G-ORANGE',
          'o5G-FREE MOBILE', 'o5G-BOUYGUES TELECOM', 'o5G-SFR', 'o5G-ORANGE']

for a in A:
    for C in C_space:  # tuples de 0/1
        for O1 in O_full:    # toutes les offres existantes
            for O2 in O_full:                                               
                if (a, C, O1, O2) not in f_data:
                   f_data[a, C, O1, O2] = 0

for a in A:
    for C in C_space:  
        for O1 in O_full:
            v = 1
            for O2 in O_full:
                if O2 != O1 :
                    v -= f_data[a, C, O1, O2]
            f_data[a, C, O1, O1] = v                             #tous ceux qui ne bougent pas restent chez eux.



model.f = pyo.Param(model.A, model.Cvec, model.O, model.O, initialize=f_data) # taux de migration

#un grand M pour les contraintes de Big-M
# Tight lower/upper bounds for sigma[t,a,i,o]
# sigma[t,a,i,o] = sum((f[a,C1,o_prev,o] - f[a,C0,o_prev,o]) * u[t-1,a,i_prev,o_prev])

L_data = {}
U_data = {}

eps = 1e-4

for a in model.A:
    ua = pyo.value(model.u_a[a])

    for t in model.T:
        C0 = _C_tuple(model, t, a, 0)
        C1 = _C_tuple(model, t, a, 1)

        for o in model.O:
            coeffs = []

            for i_prev in model.I:
                for o_prev in model.O_i[i_prev]:
                    d = pyo.value(model.f[a, C1, o_prev, o]) - pyo.value(model.f[a, C0, o_prev, o])
                    coeffs.append(d)

            d_max = max(coeffs)
            d_min = min(coeffs)

            # Using conservation of total users in each area:
            # sum_{i_prev,o_prev} u[t-1,a,i_prev,o_prev] = u_a[a], u >= 0
            U_data[(a, o, t)] = max(0.0, d_max) * ua + eps
            L_data[(a, o, t)] = min(0.0, d_min) * ua - eps

model.U_sigma = pyo.Param(model.A, model.O, model.T, initialize=U_data, mutable=True)
model.L_sigma = pyo.Param(model.A, model.O, model.T, initialize=L_data, mutable=True)


In [29]:
# 3. VARIABLES

model.z = pyo.Var(model.T, model.S, within=pyo.Binary)
model.r = pyo.Var(model.T, model.A, within=pyo.Binary)

# Variable de linéarisation S (16)
model.S_var = pyo.Var(model.T, model.A, model.I, model.O, within=pyo.Reals)

model.u = pyo.Var(model.T, model.A, model.I, model.O, within=pyo.NonNegativeReals)
model.u_site = pyo.Var(model.T, model.A, model.S, within=pyo.NonNegativeReals)  # on ne définit pas l'opérateur et l'offre car on ne parle que de notre opérateur et la NG

In [30]:
# 4. CONTRAINTES
# 4. CONTRAINTES

# (2) r_ta ≤ sum_s z_ts
def coverage_upper(m, t, a):
    return m.r[t, a] <= sum(m.z[t, s] for s in Sa_dict[a])
model.c_2 = pyo.Constraint(model.T, model.A, rule=coverage_upper)

# (3) z_ts ≤ r_ta  ∀ s couvrant a
def coverage_lower(m, t, s, a):
    if a in As_dict[s]:
        return m.z[t, s] <= m.r[t, a]
    return pyo.Constraint.Skip
model.c_3 = pyo.Constraint(model.T, model.S, model.A, rule=coverage_lower)


# (4) Évolution des utilisateurs
def u_eq_rule(m, t, a, i, o):
    if o in O_i[i]:
        if t == min(m.T):
            return m.u[t, a, i, o] == m.ua0[a, i, o]

        C0 = _C_tuple(m, t, a, 0)

        migration_f0 = sum(
        m.f[a, C0, o_prev, o] * m.u[t-1, a, i_prev, o_prev]
        for i_prev in m.I for o_prev in m.O_i[i_prev]
    )

        return m.u[t, a, i, o] == m.S_var[t, a, i, o] + migration_f0
    else:
        return m.u[t, a, i, o] == 0.0

model.c_4 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=u_eq_rule)

# (5) Expression de sigma (pour la linéarisation de S)
def sigma_rule(m, t, a, i, o):
    if t == min(m.T):
        return 0.0
    C0 = _C_tuple(m, t, a, 0) # r = 0
    C1 = _C_tuple(m, t, a, 1) # r = 1
    
    return sum((m.f[a, C1, o_prev, o] - m.f[a, C0, o_prev, o]) * m.u[t-1, a, i_prev, o_prev]
               for i_prev in m.I for o_prev in m.O_i[i_prev])
model.sigma = pyo.Expression(model.T, model.A, model.I, model.O, rule=sigma_rule)

# S_var = r * sigma with asymmetric bounds:
# L <= sigma <= U

def c_6_rule(m, t, a, i, o):
    if t == min(m.T) or o not in O_i[i]:
        return pyo.Constraint.Skip
    U = m.U_sigma[a, o, t]
    return m.S_var[t, a, i, o] <= U * m.r[t, a]
model.c_6 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_6_rule)

def c_7_rule(m, t, a, i, o):
    if o not in O_i[i]:
        return m.S_var[t, a, i, o] == 0.0
    if t == min(m.T):
        return m.S_var[t, a, i, o] == 0.0
    U = m.U_sigma[a, o, t]
    return m.S_var[t, a, i, o] <= m.r[t, a] * U

model.c_7 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_7_rule)

def c_8_rule(m, t, a, i, o):
    if t == min(m.T) or o not in O_i[i]:
        return pyo.Constraint.Skip
    L = m.L_sigma[a, o, t]
    return m.S_var[t, a, i, o] <= m.sigma[t, a, i, o] - L * (1 - m.r[t, a])
model.c_8_sigma = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_8_rule)

def c_9_rule(m, t, a, i, o):
    if t == min(m.T) or o not in O_i[i]:
        return pyo.Constraint.Skip
    U = m.U_sigma[a, o, t]
    return m.S_var[t, a, i, o] >= m.sigma[t, a, i, o] - U * (1 - m.r[t, a])
model.c_9 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_9_rule)


# (10) u_NO = somme sur les sites
def assign_users(m, t, a):
    return m.u[t, a, τ, NG] == sum(m.u_site[t, a, s] for s in Sa_dict[a])
model.c_10 = pyo.Constraint(model.T, model.A, rule=assign_users)


# (11) Capacité du réseau
def capacity(m, t, s):
    return sum(model.DNG[t] * m.u_site[t, a, s] for a in As_dict[s]) <= model.CAPANG[t] * m.z[t, s]
model.c_11 = pyo.Constraint(model.T, model.S, rule=capacity)

# (8) budget sur le nombre de sites déployés par période
def limit_z(m, t):
    if t == min(m.T):
        return sum(m.z[t, s] for s in model.S) <= model.Zmax[t]
    return sum(m.z[t, s] - m.z[t-1, s] for s in model.S) <= model.Zmax[t]

model.c_15 = pyo.Constraint(model.T, rule=limit_z)

# (12) Budget sur le nombre de sites déployés par période
def limit_z(m, t):
    if t == min(m.T):
        return sum(m.z[t, s] for s in m.S) <= m.Zmax[t]
    return sum(m.z[t, s] - m.z[t-1, s] for s in m.S) <= m.Zmax[t]
model.c_12 = pyo.Constraint(model.T, rule=limit_z)

# (13) Couverture population minimale
def cov_pop(m, t):
    return sum(m.u_a[a] * m.r[t, a] for a in m.A) >= m.QA[t] * sum(m.u_a[a] for a in m.A)
model.c_13 = pyo.Constraint(model.T, rule=cov_pop)

# La contrainte de la crossance des z_st
def growth_z(m, t, s):
    if t == min(m.T):
        return pyo.Constraint.Skip
    return m.z[t, s]>= m.z[t-1, s]
model.c_growth_z = pyo.Constraint(model.T, model.S, rule=growth_z)

In [31]:
# 5. OBJECTIF

def objective(m):
    T_end = max(m.T)
    return sum(m.u[T_end, a, τ, NG] for a in m.A)
model.obj = pyo.Objective(rule=objective, sense=pyo.maximize)


def solve_fix_and_relax(model, W=2, solver_name='glpk'):
    """
    Matheuristique Fix-and-Relax avec propagation de monotonie pour Pyomo.
    Garde la compatibilité peu importe les données de T, S, et A en entrée.
    """
    solver = pyo.SolverFactory(solver_name)
    
    # 1. Sécuriser les bornes continues des variables pour la relaxation
    for t in model.T:
        for s in model.S:
            model.z[t, s].setlb(0)
            model.z[t, s].setub(1)
        for a in model.A:
            model.r[t, a].setlb(0)
            model.r[t, a].setub(1)
            
    # On s'assure que T est trié chronologiquement (ex: [0, 1, 2, ...])
    T_list = sorted(list(model.T))
    
    # Dictionnaires pour stocker les décisions définitives : (t, s) -> valeur
    fixed_z = {} 
    fixed_r = {} 
    
    print("==================================================")
    print(f" DÉBUT DU FIX-AND-RELAX (Fenêtre W={W})")
    print("==================================================")
    
    # Boucle glissante sur le temps
    for t_start_idx in range(len(T_list)):
        t_start = T_list[t_start_idx]
        window_end_idx = min(t_start_idx + W - 1, len(T_list) - 1)
        
        print(f"\n--- Itération pour l'instant t_start = {t_start} ---")
        
        # 2. Mise à jour des domaines (Binaire vs Continu) et Fixation
        for t_idx, t in enumerate(T_list):
            
            # Gestion des variables z (Sites)
            for s in model.S:
                model.z[t, s].unfix() # On débloque la variable par sécurité
                
                if (t, s) in fixed_z:
                    # Si la décision a déjà été prise dans le passé (ou via propagation)
                    model.z[t, s].fix(fixed_z[(t, s)])
                elif t_start_idx <= t_idx <= window_end_idx:
                    # Dans la fenêtre : variable binaire
                    model.z[t, s].domain = pyo.Binary
                else:
                    # Dans le futur : relaxation continue
                    model.z[t, s].domain = pyo.Reals
                    
            # Gestion des variables r (Zones de couverture)
            for a in model.A:
                model.r[t, a].unfix()
                
                if (t, a) in fixed_r:
                    model.r[t, a].fix(fixed_r[(t, a)])
                elif t_start_idx <= t_idx <= window_end_idx:
                    model.r[t, a].domain = pyo.Binary
                else:
                    model.r[t, a].domain = pyo.Reals
                    
        # 3. Résolution du sous-problème
        # tee=False permet de ne pas polluer l'affichage de la console avec les logs GLPK à chaque boucle
        results = solver.solve(model, tee=False) 
        
        if results.solver.termination_condition != pyo.TerminationCondition.optimal:
            print(f"⚠️ Attention : Le solveur a rencontré une difficulté à l'étape {t_start}.")
            
        # 4. Enregistrement des décisions et Propagation Avant (Monotonie)
        for s in model.S:
            val_z = round(pyo.value(model.z[t_start, s]))
            fixed_z[(t_start, s)] = val_z
            
            # LA MAGIE DE LA PROPAGATION : Si on allume (1), ça reste allumé pour toujours !
            if val_z == 1:
                for future_t_idx in range(t_start_idx + 1, len(T_list)):
                    future_t = T_list[future_t_idx]
                    fixed_z[(future_t, s)] = 1
                    
        for a in model.A:
            val_r = round(pyo.value(model.r[t_start, a]))
            fixed_r[(t_start, a)] = val_r
            
            # Propagation de la couverture (Si on couvre à t, on couvre à t+1)
            if val_r == 1:
                for future_t_idx in range(t_start_idx + 1, len(T_list)):
                    future_t = T_list[future_t_idx]
                    fixed_r[(future_t, a)] = 1

        print(f"Objectif intermédiaire estimé : {pyo.value(model.obj)}")

    # 5. Dernier passage : Consolidation des variables continues (flux u, S_var)
    print("\n--- Résolution Finale (Toutes les décisions binaires figées) ---")
    for t in model.T:
        for s in model.S:
            if (t, s) in fixed_z:
                model.z[t, s].fix(fixed_z[(t, s)])
        for a in model.A:
            if (t, a) in fixed_r:
                model.r[t, a].fix(fixed_r[(t, a)])
                
    final_results = solver.solve(model, tee=False)
    
    print("==================================================")
    print(f" FIN DU FIX-AND-RELAX | Objectif final : {pyo.value(model.obj)}")
    print("==================================================")
    
    return model

# Appel de la fonction pour lancer l'algorithme sur votre modèle
# Vous pouvez changer la taille de la fenêtre (W) selon la complexité désirée
model = solve_fix_and_relax(model, W=3, solver_name='glpk')

# # Display solver status
# print("\nSolver status:", results.solver.status)
# print("Termination condition:", results.solver.termination_condition)

# 1) Valeur de l'objectif
print("Valeur de l'objectif :", value(model.obj))

 DÉBUT DU FIX-AND-RELAX (Fenêtre W=3)

--- Itération pour l'instant t_start = 0 ---
Objectif intermédiaire estimé : 15122.434674789896

--- Itération pour l'instant t_start = 1 ---
Objectif intermédiaire estimé : 14929.357500035021

--- Itération pour l'instant t_start = 2 ---
Objectif intermédiaire estimé : 14927.210698156472

--- Itération pour l'instant t_start = 3 ---
Objectif intermédiaire estimé : 14927.211460625782

--- Itération pour l'instant t_start = 4 ---
Objectif intermédiaire estimé : 14927.211600625782

--- Itération pour l'instant t_start = 5 ---
Objectif intermédiaire estimé : 14927.211600625782

--- Résolution Finale (Toutes les décisions binaires figées) ---
 FIN DU FIX-AND-RELAX | Objectif final : 14927.211140713782
Valeur de l'objectif : 14927.211140713782


In [32]:
import time

# Nouvelle version optimisée de l'algorithme génétique sans modifier l'ancienne cellule.
def genetic_algorithm_fast(model, pop_size=30, generations=20, mutation_rate=0.01, elite_size=5):
    """Algorithme génétique optimisé pour réduire les temps de calcul."""
    random.seed(time.time())

    for t in model.T:
        for s in model.S:
            model.z[t, s].unfix()
            model.z[t, s].domain = pyo.Binary
        for a in model.A:
            model.r[t, a].unfix()
            model.r[t, a].domain = pyo.Binary

    solver = pyo.SolverFactory('glpk')
    T_list = sorted(list(model.T))
    S_list = list(model.S)
    fitness_cache = {}

    def individual_to_z(ind):
        z_dict = {}
        idx = 0
        for s in S_list:
            for t in T_list:
                z_dict[(t, s)] = ind[idx]
                idx += 1
        return z_dict

    ind_length = len(T_list) * len(S_list)

    zmax_by_t = {t: int(pyo.value(model.Zmax[t])) for t in T_list}

    def site_block_start(site_idx):
        return site_idx * len(T_list)

    def get_deploy_time(ind, site_idx):
        start = site_block_start(site_idx)
        for offset, t in enumerate(T_list):
            if ind[start + offset] == 1:
                return t
        return None

    def set_deploy_time(ind, site_idx, deploy_t):
        start = site_block_start(site_idx)
        for offset, t in enumerate(T_list):
            ind[start + offset] = 0 if deploy_t is None or t < deploy_t else 1

    def is_zmax_feasible(ind):
        new_sites = {t: 0 for t in T_list}
        for site_idx in range(len(S_list)):
            deploy_t = get_deploy_time(ind, site_idx)
            if deploy_t is not None:
                new_sites[deploy_t] += 1
        return all(new_sites[t] <= zmax_by_t[t] for t in T_list)

    def repair_individual(ind):
        deploy_times = [get_deploy_time(ind, site_idx) for site_idx in range(len(S_list))]

        for site_idx, deploy_t in enumerate(deploy_times):
            set_deploy_time(ind, site_idx, deploy_t)

        used_capacity = {t: 0 for t in T_list}
        overflow_sites = []
        for site_idx, deploy_t in enumerate(deploy_times):
            if deploy_t is None:
                continue
            if used_capacity[deploy_t] < zmax_by_t[deploy_t]:
                used_capacity[deploy_t] += 1
            else:
                deploy_times[site_idx] = None
                overflow_sites.append(site_idx)

        for site_idx in overflow_sites:
            for t in T_list:
                if t != 0 and used_capacity[t] < zmax_by_t[t]:
                    deploy_times[site_idx] = t
                    used_capacity[t] += 1
                    break

        for site_idx, deploy_t in enumerate(deploy_times):
            set_deploy_time(ind, site_idx, deploy_t)
        return ind

    def create_individual():
        ind = [0] * ind_length
        remaining_capacity = zmax_by_t.copy()
        site_indices = list(range(len(S_list)))
        random.shuffle(site_indices)
        for site_idx in site_indices:
            deploy_choices = T_list[1:] + [None]
            random.shuffle(deploy_choices)
            chosen_t = None
            for deploy_t in deploy_choices:
                if deploy_t is None or remaining_capacity[deploy_t] > 0:
                    chosen_t = deploy_t
                    break
            set_deploy_time(ind, site_idx, chosen_t)
            if chosen_t is not None:
                remaining_capacity[chosen_t] -= 1
        return repair_individual(ind)

    def evaluate_individual(ind):
        key = tuple(ind)
        if key in fitness_cache:
            return fitness_cache[key]

        if not is_zmax_feasible(ind):
            fitness_cache[key] = -1e9
            return fitness_cache[key]

        z_dict = individual_to_z(ind)
        for t in model.T:
            for s in model.S:
                model.z[t, s].fix(z_dict[(t, s)])

        results = solver.solve(model, tee=False)
        if results.solver.termination_condition == pyo.TerminationCondition.optimal:
            fit = pyo.value(model.obj)
        else:
            fit = -1e6

        for t in model.T:
            for s in model.S:
                model.z[t, s].unfix()

        fitness_cache[key] = fit
        return fit

    population = [repair_individual(create_individual()) for _ in range(pop_size)]

    for gen in range(generations):
        fitnesses = [evaluate_individual(ind) for ind in population]

        elite_indices = np.argsort(fitnesses)[-elite_size:]
        elite = [population[i] for i in elite_indices]

        new_population = elite.copy()
        while len(new_population) < pop_size:
            candidates = random.sample(range(pop_size), 3)
            winner = max(candidates, key=lambda i: fitnesses[i])
            new_population.append(population[winner].copy())

        for i in range(elite_size, len(new_population), 2):
            if i + 1 < len(new_population):
                crossover_point = random.randint(1, ind_length - 1)
                parent1 = new_population[i]
                parent2 = new_population[i + 1]
                new_population[i] = repair_individual(parent1[:crossover_point] + parent2[crossover_point:])
                new_population[i + 1] = repair_individual(parent2[:crossover_point] + parent1[crossover_point:])

        for ind in new_population[elite_size:]:
            for j in range(ind_length):
                if random.random() < mutation_rate:
                    ind[j] = 1 - ind[j]
            repair_individual(ind)

        population = new_population
        print(f"Génération {gen+1}: Meilleure fitness = {max(fitnesses)}")

    final_fitnesses = [evaluate_individual(ind) for ind in population]
    best_idx = int(np.argmax(final_fitnesses))
    best_ind = population[best_idx]
    best_z = individual_to_z(best_ind)

    for t in model.T:
        for s in model.S:
            model.z[t, s].fix(best_z[(t, s)])

    print(f"Fitness finale : {final_fitnesses[best_idx]}")
    return model

# Exemple d'utilisation de la version rapide
model_optimized_fast = genetic_algorithm_fast(model, pop_size=30, generations=15, mutation_rate=0.01, elite_size=4)

Génération 1: Meilleure fitness = 11351.32998974228
Génération 2: Meilleure fitness = 13572.493821246384
Génération 3: Meilleure fitness = 13764.121236286459
Génération 4: Meilleure fitness = 13985.347568679595
Génération 5: Meilleure fitness = 14031.544235439635
Génération 6: Meilleure fitness = 14203.528968625411
Génération 7: Meilleure fitness = 14211.67359647446
Génération 8: Meilleure fitness = 14401.658136023572
Génération 9: Meilleure fitness = 14401.658136023572
Génération 10: Meilleure fitness = 14589.63795694989
Génération 11: Meilleure fitness = 14589.63795694989
Génération 12: Meilleure fitness = 14589.63795694989
Génération 13: Meilleure fitness = 14597.131871472844
Génération 14: Meilleure fitness = 14626.419853554267
Génération 15: Meilleure fitness = 14626.419853554267
Fitness finale : 14713.839851867444


In [33]:
# Combinaison Hybride : Fix-and-Relax suivi d'Algorithme Génétique

import time

def hybrid_fix_relax_ga(model, W=2, pop_size=20, generations=15, mutation_rate=0.01, elite_size=5):
    """
    Méthode hybride combinant Fix-and-Relax et Algorithme Génétique.
    Commence par Fix-and-Relax pour obtenir une solution de base, puis utilise l'AG pour l'améliorer.
    
    Paramètres:
    - model: le modèle Pyomo
    - W: taille de la fenêtre pour Fix-and-Relax
    - pop_size, generations, mutation_rate, elite_size: paramètres pour l'AG
    
    Retourne: le modèle avec la meilleure solution trouvée
    """
    
    # Étape 1: Exécuter Fix-and-Relax pour obtenir une solution initiale
    print("=== Étape 1: Fix-and-Relax ===")
    
    model_fr = solve_fix_and_relax(model, W=W)
    fr_objective = pyo.value(model_fr.obj)
    print(f"Objectif après Fix-and-Relax: {fr_objective}")
    
    # Étape 2: Extraire la solution de Fix-and-Relax
    T_list = sorted(list(model.T))
    S_list = list(model.S)
    fixed_z = {}
    for t in T_list:
        for s in S_list:
            fixed_z[(t, s)] = int(pyo.value(model_fr.z[t, s]))
    
    # Convertir en individu (vecteur binaire)
    ind_fixed = []
    for s in S_list:
        for t in T_list:
            ind_fixed.append(fixed_z[(t, s)])
    
    # Étape 3: Initialiser la population de l'AG avec des variations de la solution Fix-and-Relax
    print("\n=== Étape 2: Initialisation de l'AG avec la solution Fix-and-Relax ===")
    random.seed(time.time())
    
    zmax_by_t = {t: int(pyo.value(model.Zmax[t])) for t in T_list}

    def site_block_start(site_idx):
        return site_idx * len(T_list)

    def get_deploy_time(ind, site_idx):
        start = site_block_start(site_idx)
        for offset, t in enumerate(T_list):
            if ind[start + offset] == 1:
                return t
        return None

    def set_deploy_time(ind, site_idx, deploy_t):
        start = site_block_start(site_idx)
        for offset, t in enumerate(T_list):
            ind[start + offset] = 0 if deploy_t is None or t < deploy_t else 1

    def is_zmax_feasible(ind):
        new_sites = {t: 0 for t in T_list}
        for site_idx in range(len(S_list)):
            deploy_t = get_deploy_time(ind, site_idx)
            if deploy_t is not None:
                new_sites[deploy_t] += 1
        return all(new_sites[t] <= zmax_by_t[t] for t in T_list)

    def repair_individual(ind):
        deploy_times = [get_deploy_time(ind, site_idx) for site_idx in range(len(S_list))]

        for site_idx, deploy_t in enumerate(deploy_times):
            set_deploy_time(ind, site_idx, deploy_t)

        used_capacity = {t: 0 for t in T_list}
        overflow_sites = []
        for site_idx, deploy_t in enumerate(deploy_times):
            if deploy_t is None:
                continue
            if used_capacity[deploy_t] < zmax_by_t[deploy_t]:
                used_capacity[deploy_t] += 1
            else:
                deploy_times[site_idx] = None
                overflow_sites.append(site_idx)

        for site_idx in overflow_sites:
            for t in T_list:
                if t != 0 and used_capacity[t] < zmax_by_t[t]:
                    deploy_times[site_idx] = t
                    used_capacity[t] += 1
                    break

        for site_idx, deploy_t in enumerate(deploy_times):
            set_deploy_time(ind, site_idx, deploy_t)
        return ind
    
    def create_individual_from_fixed(fixed_ind, mutation_prob=0.1):
        """Crée un individu basé sur la solution Fix-and-Relax avec quelques mutations pour la diversité."""
        ind = fixed_ind.copy()
        for j in range(len(ind)):
            if random.random() < mutation_prob:
                ind[j] = 1 - ind[j]
        repair_individual(ind)
        return ind
    
    population = [create_individual_from_fixed(ind_fixed) for _ in range(pop_size - 1)]
    population.append(repair_individual(ind_fixed.copy()))  # Inclure la solution originale dans la population
    
    # Étape 4: Préparer l'AG
    for t in model.T:
        for s in model.S:
            model.z[t, s].unfix()
            model.z[t, s].domain = pyo.Binary
        for a in model.A:
            model.r[t, a].unfix()
            model.r[t, a].domain = pyo.Binary
    
    solver = pyo.SolverFactory('glpk')
    fitness_cache = {}
    ind_length = len(T_list) * len(S_list)
    
    def individual_to_z(ind):
        z_dict = {}
        idx = 0
        for s in S_list:
            for t in T_list:
                z_dict[(t, s)] = ind[idx]
                idx += 1
        return z_dict
    
    def evaluate_individual(ind):
        key = tuple(ind)
        if key in fitness_cache:
            return fitness_cache[key]
        
        if not is_zmax_feasible(ind):
            fitness_cache[key] = -1e9
            return fitness_cache[key]

        z_dict = individual_to_z(ind)
        for t in model.T:
            for s in model.S:
                model.z[t, s].fix(z_dict[(t, s)])
        
        results = solver.solve(model, tee=False)
        if results.solver.termination_condition == pyo.TerminationCondition.optimal:
            fit = pyo.value(model.obj)
        else:
            fit = -1e6
        
        for t in model.T:
            for s in model.S:
                model.z[t, s].unfix()
        
        fitness_cache[key] = fit
        return fit
    
    # Étape 5: Boucle de l'AG
    print("=== Étape 3: Exécution de l'Algorithme Génétique ===")
    for gen in range(generations):
        fitnesses = [evaluate_individual(ind) for ind in population]
        
        elite_indices = np.argsort(fitnesses)[-elite_size:]
        elite = [population[i] for i in elite_indices]
        
        new_population = elite.copy()
        while len(new_population) < pop_size:
            candidates = random.sample(range(pop_size), 3)
            winner = max(candidates, key=lambda i: fitnesses[i])
            new_population.append(population[winner].copy())
        
        for i in range(elite_size, len(new_population), 2):
            if i + 1 < len(new_population):
                crossover_point = random.randint(1, ind_length - 1)
                parent1 = new_population[i]
                parent2 = new_population[i + 1]
                new_population[i] = repair_individual(parent1[:crossover_point] + parent2[crossover_point:])
                new_population[i + 1] = repair_individual(parent2[:crossover_point] + parent1[crossover_point:])
        
        for ind in new_population[elite_size:]:
            for j in range(ind_length):
                if random.random() < mutation_rate:
                    ind[j] = 1 - ind[j]
            repair_individual(ind)
        
        population = new_population
        print(f"Génération {gen+1}: Meilleure fitness = {max(fitnesses)}")
    
    # Étape 6: Sélection de la meilleure solution
    final_fitnesses = [evaluate_individual(ind) for ind in population]
    best_idx = int(np.argmax(final_fitnesses))
    best_ind = population[best_idx]
    best_z = individual_to_z(best_ind)
    
    for t in model.T:
        for s in model.S:
            model.z[t, s].fix(best_z[(t, s)])
    
    hybrid_objective = final_fitnesses[best_idx]
    print(f"\nObjectif final hybride: {hybrid_objective}")
    print(f"Amélioration par rapport à Fix-and-Relax: {hybrid_objective - fr_objective}")
    
    return model

# Exemple d'utilisation de la méthode hybride
model_hybrid = hybrid_fix_relax_ga(model, W=2, pop_size=20, generations=10, mutation_rate=0.01, elite_size=4)

=== Étape 1: Fix-and-Relax ===
 DÉBUT DU FIX-AND-RELAX (Fenêtre W=2)

--- Itération pour l'instant t_start = 0 ---
Objectif intermédiaire estimé : 16733.256285983964

--- Itération pour l'instant t_start = 1 ---
Objectif intermédiaire estimé : 15122.434674789896

--- Itération pour l'instant t_start = 2 ---
Objectif intermédiaire estimé : 14929.358540316467

--- Itération pour l'instant t_start = 3 ---
Objectif intermédiaire estimé : 14927.211558569681

--- Itération pour l'instant t_start = 4 ---
Objectif intermédiaire estimé : 14927.211600625782

--- Itération pour l'instant t_start = 5 ---
Objectif intermédiaire estimé : 14927.211600625782

--- Résolution Finale (Toutes les décisions binaires figées) ---
 FIN DU FIX-AND-RELAX | Objectif final : 14927.211140713782
Objectif après Fix-and-Relax: 14927.211140713782

=== Étape 2: Initialisation de l'AG avec la solution Fix-and-Relax ===
=== Étape 3: Exécution de l'Algorithme Génétique ===
Génération 1: Meilleure fitness = 14927.211600625